# TSBC MaritimeBench — Google Colab Execution Notebook
### Reproducible Orchestration for the 18-Stage Maritime Accident Pipeline
**Version**: 2.1 (Production Research Pipeline)  
**Target Environment**: Google Colab (with GPU acceleration & Google Drive persistence)

---

### Overview & Architecture
This notebook provides an automated, reproducible, checkpointed orchestration layer around the scientific pipeline repository `TSBC-MaritimePipeline-PR-main`.

* **18 End-to-End Stages**: Covers raw MARSIS relational ingestion, record validation, domain-informed document generation, corpus export, linguistic/importance profiling, multi-model masked language model (MLM) and pseudo-log-likelihood (PLL) benchmarking, statistical validation, and automated multi-criteria decision analysis (MCDA).
* **Zero Modification to Scientific Code**: The underlying algorithms, tokenizer evaluations, perplexity metrics, statistical tests, and decision logic remain identical to the peer-reviewed codebase.
* **Persistent Google Drive Storage**: All outputs, intermediate checkpoints, model caches, logs, and experiment metadata persist across ephemeral Colab session restarts.
* **Resilient Stage Runner**: Supports granular step-by-step execution, stage ranges (e.g., `01` to `18`), stage skipping, interrupted run resumption, and GPU memory cleanup.

## 0. Research Run Configuration

Configure the experiment parameters for this execution. You can control:
1. **`RUN_MODE`**:
   - `"preserve"` (Default): Keeps existing experimental outputs intact.
   - `"archive_then_run"`: Archives current `outputs/` before executing.
   - `"clean_run"`: Explicitly removes prior outputs for a fresh run.
2. **`RUN_FROM_STAGE` / `RUN_TO_STAGE`**: Range of stages to execute (e.g., `"01"` to `"18"` or `"13"` to `"17"`).
3. **`RESUME_FROM_STAGE`**: Set to resume an interrupted run (e.g., `"14"`). Prior stage outputs will be verified.
4. **`SKIP_COMPLETED_STAGES`**: When `True`, stages whose output artifacts already exist will be skipped. Defaults to `False` for strict reproducibility.
5. **`DATA_SOURCE_DIR`**: Optional Google Drive path containing raw MARSIS CSV files.

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

# ==============================================================================
# TSBC MARITIME BENCHMARK — RESEARCH RUN CONFIGURATION
# ==============================================================================

# 1. Pipeline Execution Mode:
#    - "preserve"         : Keep existing outputs intact (Default & Recommended)
#    - "archive_then_run" : Backup current outputs to an archive folder before running
#    - "clean_run"        : Explicitly wipe generated stage outputs before running
RUN_MODE = "preserve"

# 2. Stage Execution Selection:
#    - Set RUN_FROM_STAGE and RUN_TO_STAGE to define an execution range
#    - Stage keys: "01", "02", "03", "04", "05", "05a", "06", "07", "08", "09",
#                  "10", "11", "12", "13", "14", "15", "16", "17", "18"
RUN_FROM_STAGE = "01"
RUN_TO_STAGE = "18"

# 3. Interruption & Resumption:
#    - Set RESUME_FROM_STAGE (e.g. "14") to resume from that stage onward without re-running prior stages.
#    - The notebook verifies that required prior stage artifacts exist before starting.
RESUME_FROM_STAGE = None  # e.g., "14" or None

# 4. Safe Skip Mode:
#    - If True, stages whose output artifacts already exist will be skipped.
#    - Default is False to guarantee scientific reproducibility.
SKIP_COMPLETED_STAGES = False

# 5. Project Repository Setup:
#    - Set RESET_PROJECT = True only if you wish to re-extract the repository ZIP from scratch.
RESET_PROJECT = False

# 6. Raw MARSIS Data Location:
#    - If your raw CSV files reside in a Google Drive folder, specify the path here:
#      e.g., "/content/drive/MyDrive/MARSIS_RAW_DATA"
#    - Leave as "" if uploading CSVs directly through the notebook.
DATA_SOURCE_DIR = ""

# 7. Persistent Storage Paths (Google Drive):
DRIVE_ROOT = "/content/drive/MyDrive/TSBC-MaritimeBench"
PROJECT_ROOT = f"{DRIVE_ROOT}/TSBC-MaritimePipeline-PR-main"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ARCHIVE_DIR = f"{DRIVE_ROOT}/archived_runs/{RUN_ID}"
RUN_LOGS_DIR = f"{DRIVE_ROOT}/run_logs"
CHECKPOINTS_DIR = f"{DRIVE_ROOT}/checkpoints"
MODEL_CACHE_DIR = f"{DRIVE_ROOT}/model_cache"

# Hugging Face persistent model cache paths on Google Drive
HF_HOME = f"{MODEL_CACHE_DIR}/huggingface"
TRANSFORMERS_CACHE = f"{HF_HOME}/transformers"
HF_DATASETS_CACHE = f"{HF_HOME}/datasets"

print(f"======================================================================")
print(f" TSBC MARITIME BENCHMARK — RUN CONFIGURATION INITIALIZED")
print(f"======================================================================")
print(f"Run ID:                 {RUN_ID}")
print(f"Run Mode:               {RUN_MODE}")
print(f"Execution Range:        {RUN_FROM_STAGE} -> {RUN_TO_STAGE}")
print(f"Resume Stage:           {RESUME_FROM_STAGE}")
print(f"Skip Completed Stages:  {SKIP_COMPLETED_STAGES}")
print(f"Drive Root:             {DRIVE_ROOT}")
print(f"Project Root:           {PROJECT_ROOT}")
print(f"Run Archive Dir:        {RUN_ARCHIVE_DIR}")
print(f"Hugging Face Cache:     {HF_HOME}")
print(f"======================================================================")

## 1. Google Drive and Environment Setup

Mount Google Drive to establish persistent storage. Because Colab instances are ephemeral, all benchmark outputs, checkpoints, model caches, logs, and experiment metadata are persisted directly in Google Drive under:

```text
TSBC-MaritimeBench/
    TSBC-MaritimePipeline-PR-main/
    checkpoints/
    model_cache/
    run_logs/
    archived_runs/
```

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("Mounting Google Drive to /content/drive...")
        drive.mount('/content/drive')
        print("[SUCCESS] Google Drive mounted successfully.")
    else:
        print("[INFO] Google Drive is already mounted.")
except ImportError:
    print("[NOTE] Running outside Google Colab environment. Skipping drive.mount().")

# 2. Create Required Persistent Directory Hierarchy
required_dirs = [
    DRIVE_ROOT,
    PROJECT_ROOT,
    CHECKPOINTS_DIR,
    MODEL_CACHE_DIR,
    RUN_LOGS_DIR,
    f"{DRIVE_ROOT}/archived_runs",
    RUN_ARCHIVE_DIR,
    HF_HOME,
    TRANSFORMERS_CACHE,
    HF_DATASETS_CACHE
]

print("\nVerifying persistent directories:")
for d in required_dirs:
    os.makedirs(d, exist_ok=True)
    status = "EXISTS" if os.path.isdir(d) else "FAILED"
    print(f"  [{status}] {d}")

# 3. Check Free Disk Space
if hasattr(shutil, "disk_usage"):
    total, used, free = shutil.disk_usage(DRIVE_ROOT if os.path.exists(DRIVE_ROOT) else ".")
    print(f"\nGoogle Drive Storage Status:")
    print(f"  Total Capacity: {total / (1024**3):.2f} GB")
    print(f"  Used Storage:   {used / (1024**3):.2f} GB")
    print(f"  Free Available: {free / (1024**3):.2f} GB")

## 2. Repository Extraction / Validation

This section locates and extracts the `TSBC-MaritimePipeline-PR-main.zip` package into the persistent project directory.

* Automatically detects existing repository checkouts and preserves them when `RESET_PROJECT=False`.
* Automatically normalizes nested archive paths (e.g. `TSBC-MaritimePipeline-PR-main/TSBC-MaritimePipeline-PR-main/`).
* Verifies essential repository files (`run_pipeline.py`, `config/config.json`, `scripts/pipeline_utils.py`, `templates/`, `outputs/`).

In [ ]:
import os
import sys
import glob
import shutil
import zipfile
from pathlib import Path

repo_zip_name = "TSBC-MaritimePipeline-PR-main.zip"
candidate_zip_paths = [
    f"/content/{repo_zip_name}",
    f"{DRIVE_ROOT}/{repo_zip_name}",
    f"/content/drive/MyDrive/{repo_zip_name}",
    repo_zip_name
]

def find_repo_zip():
    for p in candidate_zip_paths:
        if os.path.exists(p):
            return p
    # Search drive root for any matching zip
    if os.path.exists(DRIVE_ROOT):
        found = glob.glob(f"{DRIVE_ROOT}/*Maritime*.zip")
        if found:
            return found[0]
    return None

project_path = Path(PROJECT_ROOT)
repo_needs_extraction = True

if project_path.exists() and (project_path / "run_pipeline.py").exists():
    if not RESET_PROJECT:
        print(f"[INFO] Existing repository found at {PROJECT_ROOT}.")
        print("[INFO] RESET_PROJECT is False. Preserving existing repository structure and outputs.")
        repo_needs_extraction = False
    else:
        print(f"[WARNING] RESET_PROJECT is True! Reinitializing repository at {PROJECT_ROOT}.")
        repo_needs_extraction = True

if repo_needs_extraction:
    zip_path = find_repo_zip()
    if not zip_path:
        print(f"[ACTION REQUIRED] '{repo_zip_name}' was not detected automatically.")
        print("Please upload your repository ZIP file using the file picker below:")
        try:
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded.keys():
                if fname.endswith(".zip"):
                    zip_path = os.path.abspath(fname)
                    dest_zip = f"{DRIVE_ROOT}/{repo_zip_name}"
                    shutil.copy2(zip_path, dest_zip)
                    print(f"Saved uploaded zip to persistent storage: {dest_zip}")
                    zip_path = dest_zip
                    break
        except ImportError:
            raise FileNotFoundError(f"Could not find repository ZIP and google.colab.files is unavailable.")

    if not zip_path or not os.path.exists(zip_path):
        raise FileNotFoundError(f"Cannot proceed without {repo_zip_name}. Please place it in {DRIVE_ROOT} or /content.")

    print(f"\nExtracting repository from: {zip_path}...")
    temp_extract_dir = f"/content/temp_extract_{RUN_ID}"
    os.makedirs(temp_extract_dir, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_extract_dir)
    print("Extracted archive to temporary staging folder.")

    # Detect folder structure inside the extracted zip
    extracted_items = os.listdir(temp_extract_dir)
    source_folder = temp_extract_dir
    
    # Check for single top-level directory and normalize accidental double nesting
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(temp_extract_dir, extracted_items[0])):
        nested = os.path.join(temp_extract_dir, extracted_items[0])
        sub_items = os.listdir(nested)
        if len(sub_items) == 1 and os.path.isdir(os.path.join(nested, sub_items[0])) and (os.path.exists(os.path.join(nested, sub_items[0], "run_pipeline.py"))):
            source_folder = os.path.join(nested, sub_items[0])
        else:
            source_folder = nested
    elif os.path.exists(os.path.join(temp_extract_dir, "run_pipeline.py")):
        source_folder = temp_extract_dir

    print(f"Normalized repository root source: {source_folder}")
    
    # Copy files into PROJECT_ROOT preserving existing outputs if present
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    for item in os.listdir(source_folder):
        s = os.path.join(source_folder, item)
        d = os.path.join(PROJECT_ROOT, item)
        if item == "outputs" and os.path.exists(d) and not RESET_PROJECT:
            print("Preserving existing outputs directory during extraction.")
            continue
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
            
    # Clean up temp staging folder
    shutil.rmtree(temp_extract_dir, ignore_errors=True)
    print(f"[SUCCESS] Repository staged into {PROJECT_ROOT}")

# Verification of repository structure
print("\n--- Verifying Repository Structure ---")
essential_files = [
    "run_pipeline.py",
    "config/config.json",
    "scripts/pipeline_utils.py",
    "scripts/01_parse_dictionary.py",
    "scripts/18_lint_corpus.py",
    "templates/vessel_templates.json"
]
all_essential_found = True
for f in essential_files:
    fp = os.path.join(PROJECT_ROOT, f)
    exists = os.path.exists(fp)
    status = "FOUND" if exists else "MISSING"
    print(f"  [{status}] {f}")
    if not exists:
        all_essential_found = False

if not all_essential_found:
    raise RuntimeError("Repository validation failed! Critical pipeline files are missing.")
print("[SUCCESS] Repository structure successfully verified.")

## 3. Raw MARSIS Data Setup

The repository ZIP does not contain raw relational CSV datasets due to data governance and size constraints. The pipeline expects raw CSV files in:

```text
PROJECT_ROOT/data/
```

Required MARSIS tables detected by the pipeline:
1. `MDOTW_VW_OCCURRENCE_VESSEL_PUBLIC` (Vessel occurrences)
2. `MDOTW_VW_OCCURRENCE_PUBLIC` (Occurrence root records)
3. `MDOTW_VW_OCCURRENCE_VESSEL_REC_EQUIPMENT_PUBLIC` (Recovery equipment)
4. `MDOTW_VW_INJURIES_PUBLIC` (Injuries & fatalities)
5. `MDOTW_VW_OCCURRENCE_VESSEL_LSA_EQUIPMENT_PUBLIC` (Life-saving appliances)
6. `MDOTW_VW_OCCURRENCE_VESSEL_NAV_EQUIPMENT_PUBLIC` (Navigation equipment)
7. MARSIS Data Dictionary / Inventory CSV

This section copies raw CSVs from `DATA_SOURCE_DIR` (if set) or provides a file uploader, followed by preflight validation.

In [ ]:
import os
import sys
import shutil
import glob
from pathlib import Path

# ==============================================================================
# RAW MARSIS DATA INGESTION
# ==============================================================================
data_dir = Path(PROJECT_ROOT) / "data"
data_dir.mkdir(parents=True, exist_ok=True)

print(f"Target Data Directory: {data_dir}")

# Mechanism B: Copy from Google Drive source directory if provided
if DATA_SOURCE_DIR and os.path.exists(DATA_SOURCE_DIR):
    print(f"Copying raw CSV files from DATA_SOURCE_DIR: {DATA_SOURCE_DIR}...")
    source_csvs = glob.glob(f"{DATA_SOURCE_DIR}/*.csv")
    if not source_csvs:
        print(f"[WARNING] No CSV files found in {DATA_SOURCE_DIR}!")
    else:
        for src in source_csvs:
            dst = data_dir / Path(src).name
            shutil.copy2(src, dst)
            print(f"  Copied: {Path(src).name}")

# Mechanism A: Direct upload if data directory is empty
existing_csvs = list(data_dir.glob("*.csv"))
if not existing_csvs:
    print("\n[ACTION REQUIRED] No CSV files found in data directory.")
    print("Please upload the MARSIS CSV files and data dictionary using the file uploader below:")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".csv"):
                shutil.move(fname, str(data_dir / fname))
                print(f"  Uploaded and moved: {fname}")
    except ImportError:
        print("[NOTE] Direct upload requires Google Colab. Please copy CSV files to:", str(data_dir))

print(f"\nCurrent CSV files in {data_dir}: {[f.name for f in data_dir.glob('*.csv')]}")

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Load pipeline_utils from the project scripts
sys.path.insert(0, str(Path(PROJECT_ROOT) / "scripts"))
import pipeline_utils

data_dir = Path(PROJECT_ROOT) / "data"

EXPECTED_TABLE_STEMS = [
    ("MDOTW_VW_OCCURRENCE_VESSEL_PUBLIC", "Vessel Occurrences", True),
    ("MDOTW_VW_OCCURRENCE_PUBLIC", "Primary Occurrences", True),
    ("MDOTW_VW_OCCURRENCE_VESSEL_REC_EQUIPMENT_PUBLIC", "Recovery Equipment", True),
    ("MDOTW_VW_INJURIES_PUBLIC", "Injuries and Fatalities", True),
    ("MDOTW_VW_OCCURRENCE_VESSEL_LSA_EQUIPMENT_PUBLIC", "Life Saving Appliances", True),
    ("MDOTW_VW_OCCURRENCE_VESSEL_NAV_EQUIPMENT_PUBLIC", "Navigation Equipment", True),
]

csv_files = sorted(list(data_dir.glob("*.csv")))
print(f"======================================================================")
print(f" RAW MARSIS DATASET INVENTORY & PREFLIGHT VERIFICATION")
print(f" Data Directory: {data_dir}")
print(f" Total CSV Files Detected: {len(csv_files)}")
print(f"======================================================================")

inventory_records = []
for f in csv_files:
    size_bytes = f.stat().st_size
    size_mb = size_bytes / (1024 * 1024)
    
    # Fast row count estimation
    row_count = 0
    try:
        with open(f, 'rb') as fp:
            row_count = sum(1 for _ in fp) - 1
            if row_count < 0:
                row_count = 0
    except Exception:
        row_count = -1
        
    # Match detected role
    fname_lower = f.name.lower()
    role = "Unknown"
    if "dictionary" in fname_lower or "inventory" in fname_lower:
        role = "Data Dictionary / Inventory"
    else:
        for stem, desc, _ in EXPECTED_TABLE_STEMS:
            if stem.lower() in fname_lower or stem.replace("MDOTW_VW_", "").lower() in fname_lower:
                role = desc
                break
        if role == "Unknown":
            if "nav" in fname_lower: role = "Navigation Equipment"
            elif "rec" in fname_lower: role = "Recovery Equipment"
            elif "lsa" in fname_lower: role = "Life Saving Appliances"
            elif "injur" in fname_lower: role = "Injuries and Fatalities"
            elif "vessel" in fname_lower: role = "Vessel Occurrences"
            elif "occurrence" in fname_lower: role = "Primary Occurrences"
            
    inventory_records.append({
        "Filename": f.name,
        "Size (MB)": f"{size_mb:.2f}",
        "Est. Rows": row_count if row_count >= 0 else "N/A",
        "Detected Role": role
    })

inv_df = pd.DataFrame(inventory_records)
if not inv_df.empty:
    print(inv_df.to_string(index=False))
else:
    print("[WARNING] No CSV files detected in data directory!")

# Verify detection using canonical pipeline_utils.detect_datasets()
detection_results = pipeline_utils.detect_datasets()
detected_datasets = detection_results.get("datasets", {})
dictionary_file = detection_results.get("dictionary", None)

print("\n--- Pipeline Semantic Detection Verification ---")
missing_critical = []
for stem, desc, req in EXPECTED_TABLE_STEMS:
    match_file = detected_datasets.get(stem)
    if match_file and match_file.exists():
        print(f"  [PASS] {stem:<48} -> {match_file.name}")
    else:
        print(f"  [FAIL] {stem:<48} -> MISSING")
        if req:
            missing_critical.append(stem)

if dictionary_file and dictionary_file.exists():
    print(f"  [PASS] Data Dictionary -> {dictionary_file.name}")
else:
    print(f"  [FAIL] Data Dictionary -> MISSING")
    missing_critical.append("DATA_DICTIONARY")

if missing_critical:
    err_msg = (
        f"\n[CRITICAL ERROR] Required MARSIS dataset inputs are missing!\n"
        f"Missing items: {missing_critical}\n"
        f"The pipeline cannot run without these raw files in {data_dir}.\n"
        f"Please provide the required CSVs and re-run this validation cell."
    )
    print(err_msg)
    raise FileNotFoundError(err_msg)
else:
    print("\n[SUCCESS] All required MARSIS relational tables and data dictionary are verified and ready!")

## 4. Dependency Installation

Reads `PROJECT_ROOT/requirements.txt` and ensures scientific packages (`scipy`, `scikit-learn`, `seaborn`, `matplotlib`, `orjson`, `networkx`, `transformers`) are available while keeping Google Colab's optimized PyTorch installation intact. Diagnostics verify CUDA GPU hardware acceleration.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

req_path = Path(PROJECT_ROOT) / "requirements.txt"
if not req_path.exists():
    raise FileNotFoundError(f"requirements.txt not found at {req_path}")

print(f"Installing dependencies from {req_path} and scientific packages...")

# Colab already has PyTorch with GPU support. We install repo requirements
# along with scipy, scikit-learn, seaborn, matplotlib without downgrading PyTorch.
cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(req_path),
    "scipy>=1.10.0",
    "scikit-learn>=1.2.0",
    "seaborn>=0.12.0",
    "matplotlib>=3.7.0"
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("[WARNING] Pip installation stderr:\n", result.stderr)
else:
    print("[SUCCESS] Dependencies installed successfully.")

# PyTorch and CUDA verification
import torch
import transformers

print("\n--- Runtime Environment & GPU Diagnostics ---")
print(f"PyTorch Version:      {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = 0
    gpu_name = torch.cuda.get_device_name(device_id)
    capability = torch.cuda.get_device_capability(device_id)
    total_mem = torch.cuda.get_device_properties(device_id).total_memory / (1024**3)
    print(f"GPU Device:           {gpu_name} (Compute Capability: {capability[0]}.{capability[1]})")
    print(f"Total VRAM:           {total_mem:.2f} GB")
else:
    print("=" * 70)
    print("[PROMINENT WARNING] CUDA GPU is NOT available! Running on CPU.")
    print("Stages 14 and 15 (MLM / PLL benchmarking) will run substantially slower.")
    print("Recommendation: Navigate to Runtime -> Change runtime type -> Hardware accelerator -> GPU.")
    print("=" * 70)

## 5. GPU and Hugging Face Environment

Configure persistent Hugging Face caching directly on Google Drive (`DRIVE_ROOT/model_cache/huggingface`) to prevent redundant downloads of multi-gigabyte models (BERT, RoBERTa, DeBERTa, MaritimeBERT) across Colab restarts. Also provides GPU memory clearing utilities.

In [ ]:
import os
import gc
import torch

# Configure Hugging Face Cache on Google Drive
os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_DATASETS_CACHE
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("--- Persistent Hugging Face Environment ---")
print(f"HF_HOME:            {os.environ['HF_HOME']}")
print(f"TRANSFORMERS_CACHE: {os.environ['TRANSFORMERS_CACHE']}")
print(f"HF_DATASETS_CACHE:  {os.environ['HF_DATASETS_CACHE']}")

# Helper function to clear GPU VRAM
def clear_gpu_memory():
    """Performs aggressive garbage collection and empties CUDA memory cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        allocated = torch.cuda.memory_allocated(0) / (1024**2)
        reserved = torch.cuda.memory_reserved(0) / (1024**2)
        print(f"[GPU Memory Cleaned] Allocated: {allocated:.1f} MB | Reserved: {reserved:.1f} MB")
    else:
        print("[Memory Cleaned] Ran Python gc.collect()")

def print_gpu_stats():
    """Prints current GPU utilization and memory availability."""
    if torch.cuda.is_available():
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        allocated = torch.cuda.memory_allocated(0) / (1024**3)
        reserved = torch.cuda.memory_reserved(0) / (1024**3)
        free = total - reserved
        print(f"GPU: {torch.cuda.get_device_name(0)} | Total: {total:.2f} GB | Allocated: {allocated:.2f} GB | Free/Available: {free:.2f} GB")
    else:
        print("Device: CPU (No CUDA GPU available)")

print("\nTesting GPU cleanup utility:")
clear_gpu_memory()
print_gpu_stats()

## 6. Pipeline Preflight Validation

Runs a 13-point preflight audit before launching the pipeline:
1. Python version (>= 3.9)
2. PyTorch version
3. CUDA availability
4. GPU name & VRAM
5. Transformers library version
6. Repository file structure
7. Configuration file (`config/config.json`)
8. Templates directory (`templates/*.json`)
9. Raw MARSIS data availability & detection
10. Required scripts (`01` through `18`, `pipeline_utils.py`, `text_sanitizer.py`)
11. Existing output directories
12. Free disk space
13. Google Drive read/write accessibility

Execution will safely halt if any critical prerequisite fails.

In [ ]:
import os
import sys
import shutil
import json
from pathlib import Path

# Switch current working directory to PROJECT_ROOT
os.chdir(PROJECT_ROOT)
if str(Path(PROJECT_ROOT) / "scripts") not in sys.path:
    sys.path.insert(0, str(Path(PROJECT_ROOT) / "scripts"))

import torch
import transformers
import numpy as np
import pandas as pd
import pipeline_utils

print("Working directory:", os.getcwd())
print("Project root:     ", PROJECT_ROOT)

preflight_results = []

def record_check(name, passed, details=""):
    preflight_results.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL",
        "Details": details
    })
    return passed

# 1. Python Version
py_ver = sys.version.split()[0]
py_ok = sys.version_info >= (3, 9)
record_check("1. Python Version (>=3.9)", py_ok, f"Python {py_ver}")

# 2. PyTorch Version
torch_ver = torch.__version__
record_check("2. PyTorch Version", True, torch_ver)

# 3. CUDA Availability
cuda_ok = torch.cuda.is_available()
record_check("3. CUDA Acceleration", cuda_ok, "Available" if cuda_ok else "Not Available (CPU only)")

# 4. GPU Name & VRAM
gpu_info = torch.cuda.get_device_name(0) if cuda_ok else "N/A"
record_check("4. GPU Device", cuda_ok, gpu_info)

# 5. Transformers Version
tf_ver = transformers.__version__
record_check("5. Transformers Version", True, tf_ver)

# 6. Repository Structure
structure_ok = (
    os.path.exists("run_pipeline.py") and
    os.path.exists("config") and
    os.path.exists("scripts") and
    os.path.exists("templates")
)
record_check("6. Repository Structure", structure_ok, "Core directories present")

# 7. Config File
config_path = Path("config/config.json")
config_ok = config_path.exists()
config_details = ""
if config_ok:
    try:
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
            config_details = f"data_dir='{cfg.get('data_dir')}', output_dir='{cfg.get('output_dir')}'"
    except Exception as e:
        config_ok = False
        config_details = f"Invalid JSON: {e}"
record_check("7. Config File (config.json)", config_ok, config_details)

# 8. Templates
templates = ["vessel_templates.json", "injury_templates.json", "equipment_templates.json"]
templates_found = [t for t in templates if (Path("templates") / t).exists()]
templates_ok = len(templates_found) == len(templates)
record_check("8. Templates", templates_ok, f"{len(templates_found)}/{len(templates)} templates present")

# 9. Raw Data Availability
try:
    det = pipeline_utils.detect_datasets()
    datasets_count = len(det.get("datasets", {}))
    dict_present = det.get("dictionary") is not None
    data_ok = (datasets_count >= 6) and dict_present
    data_details = f"{datasets_count}/6 MARSIS tables mapped, Dictionary={'Found' if dict_present else 'Missing'}"
except Exception as e:
    data_ok = False
    data_details = f"Detection error: {e}"
record_check("9. Raw MARSIS Data Availability", data_ok, data_details)

# 10. Required Scripts
required_scripts = [
    "01_parse_dictionary.py", "02_profile_dataset.py", "03_discover_relationships.py",
    "04_select_semantic_columns.py", "05_merge_tables.py", "05a_validate_records.py",
    "06_generate_documents.py", "07_clean_documents.py", "08_export_corpus.py",
    "09_statistics.py", "10_extract_vocabulary.py", "11_corpus_representations.py",
    "12_semantic_importance.py", "13_tokenizer_analysis.py", "14_mlm_evaluation.py",
    "15_cross_model_benchmarking.py", "16_statistical_analysis.py", "17_decision_engine.py",
    "18_lint_corpus.py", "pipeline_utils.py", "text_sanitizer.py"
]
missing_scripts = [s for s in required_scripts if not (Path("scripts") / s).exists()]
scripts_ok = len(missing_scripts) == 0
scripts_details = "All 21 scripts found" if scripts_ok else f"Missing: {missing_scripts}"
record_check("10. Pipeline Scripts", scripts_ok, scripts_details)

# 11. Existing Output Directories
outputs_dir = Path("outputs")
outputs_ok = outputs_dir.exists()
record_check("11. Outputs Directory", True, f"Exists (Subdirectories: {len(list(outputs_dir.glob('*')))})")

# 12. Free Disk Space
total, used, free = shutil.disk_usage(PROJECT_ROOT)
free_gb = free / (1024**3)
space_ok = free_gb > 2.0
record_check("12. Free Disk Space", space_ok, f"{free_gb:.2f} GB free")

# 13. Google Drive Accessibility
drive_writable = False
test_drive_file = Path(DRIVE_ROOT) / f".test_write_{RUN_ID}.tmp"
try:
    test_drive_file.write_text("ok")
    test_drive_file.unlink()
    drive_writable = True
    drive_details = "Read/Write verified"
except Exception as e:
    drive_details = f"Write failed: {e}"
record_check("13. Google Drive Persistence", drive_writable, drive_details)

print("\n" + "=" * 80)
print(f"{'TSBC MARITIME PIPELINE PREFLIGHT VALIDATION SUMMARY':^80}")
print("=" * 80)
print(f"{'Check':<36} | {'Status':<6} | {'Details'}")
print("-" * 80)
critical_failures = []
for r in preflight_results:
    status_str = f"[{r['Status']}]"
    print(f"{r['Check']:<36} | {status_str:<6} | {r['Details']}")
    if r["Status"] == "FAIL":
        if "CUDA" not in r["Check"]:
            critical_failures.append(r["Check"])
print("=" * 80)

if critical_failures:
    err_msg = f"[CRITICAL PREFLIGHT FAILURE] The following required checks failed:\n" + "\n".join(f"  - {c}" for c in critical_failures)
    print(err_msg)
    raise RuntimeError(err_msg)
else:
    print("[SUCCESS] All critical preflight checks passed! Pipeline is ready for execution.")

## 7. Stage Runner

The orchestration engine manages the sequential execution of stages without altering any underlying research algorithms.

Key capabilities:
* **`run_stage(stage_key)`**: Executes a single stage with timing, logging, and output checkpointing.
* **`run_pipeline_range(start_stage, end_stage)`**: Runs a contiguous sequence (e.g., `"01"` to `"18"` or `"13"` to `"17"`).
* **`show_stage_outputs(stage_key)`**: Lists files generated in `outputs/stage-XX/`.
* **`inspect_stage(stage_key)`**: Previews key output files (JSON, CSV, JSONL, MD).
* **Interrupted Run Resumption**: When `RESUME_FROM_STAGE` is set, prior stages are checked for required outputs before resuming.
* **Persistent Checkpointing**: Writes execution status to `run_status.json` in `RUN_ARCHIVE_DIR` and `CHECKPOINTS_DIR`.

In [ ]:
import os
import sys
import time
import json
import shutil
import importlib
import traceback
from datetime import datetime
from pathlib import Path

# Ensure scripts directory is in sys.path
scripts_dir = str(Path(PROJECT_ROOT) / "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Canonical 18-stage definition matching run_pipeline.py
STAGES = [
    "01", "02", "03", "04", "05", "05a", "06", "07", "08", "09",
    "10", "11", "12", "13", "14", "15", "16", "17", "18"
]

STAGE_METADATA = {
    "01": ("01_parse_dictionary", "Parse Data Dictionary", "stage-01", ["dictionary_metadata.json"]),
    "02": ("02_profile_dataset", "Profile Datasets", "stage-02", ["profiling_report.json"]),
    "03": ("03_discover_relationships", "Discover Schema Relationships", "stage-03", ["relationships.json"]),
    "04": ("04_select_semantic_columns", "Select Semantic Columns", "stage-04", ["selected_semantic_columns.json"]),
    "05": ("05_merge_tables", "Merge Datasets", "stage-05", ["merged_records.jsonl"]),
    "05a": ("05a_validate_records", "Validate Records", "stage-05a", ["validation_report.json"]),
    "06": ("06_generate_documents", "Generate Natural Language Documents", "stage-06", ["raw_documents.jsonl"]),
    "07": ("07_clean_documents", "Clean and Normalize Documents", "stage-07", ["clean_documents.jsonl"]),
    "08": ("08_export_corpus", "Export Maritime Corpus & Manifest", "stage-08", ["maritime_corpus.jsonl", "manifest.json"]),
    "09": ("09_statistics", "Calculate Corpus Statistics & Report", "stage-09", ["statistics.json", "corpus_quality_report.md"]),
    "10": ("10_extract_vocabulary", "Extract Maritime Vocabulary", "stage-10", ["maritime_vocabulary.txt"]),
    "11": ("11_corpus_representations", "Multi-Format Corpus Representation Generation", "stage-11", ["corpus_representations"]),
    "12": ("12_semantic_importance", "Semantic Importance Assessment & Knowledge Classification", "stage-12", ["document_importance.jsonl"]),
    "13": ("13_tokenizer_analysis", "Multi-Model Tokenizer Benchmark Analysis", "stage-13", ["selected_models.json"]),
    "14": ("14_mlm_evaluation", "Multi-Model Masked Language Model Benchmark Matrix", "stage-14", ["pll_results.json", "bert_mlm_evaluation.json"]),
    "15": ("15_cross_model_benchmarking", "Cross-Model Benchmarking & Computational Resource Profiling", "stage-15", ["leaderboard.csv", "stage15_pareto.csv"]),
    "16": ("16_statistical_analysis", "Statistical Significance Testing & Scoring Feature Ablation", "stage-16", ["stage16_bootstrap.csv", "ablation_study.json"]),
    "17": ("17_decision_engine", "Objective Threshold Decision Engine & Research Report", "stage-17", ["decision_summary.json", "benchmark_report.md"]),
    "18": ("18_lint_corpus", "Automated Corpus Quality Linting", "stage-18", ["corpus_lint_report.json"])
}

GPU_INTENSIVE_STAGES = {"13", "14", "15"}

# Persistent Run Status State
status_file_archive = Path(RUN_ARCHIVE_DIR) / "run_status.json"
status_file_checkpoint = Path(CHECKPOINTS_DIR) / f"run_status_{RUN_ID}.json"

def load_run_status() -> dict:
    for p in [status_file_archive, status_file_checkpoint]:
        if p.exists():
            try:
                with open(p, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                pass
    return {
        "run_id": RUN_ID,
        "start_time": datetime.now().isoformat(),
        "stages": {}
    }

def save_run_status(status_dict: dict):
    status_dict["last_updated"] = datetime.now().isoformat()
    for p in [status_file_archive, status_file_checkpoint]:
        try:
            with open(p, "w", encoding="utf-8") as f:
                json.dump(status_dict, f, indent=2)
        except Exception as e:
            print(f"[WARNING] Could not save status to {p}: {e}")

# Handle Run Mode (preserve, archive_then_run, clean_run)
def apply_run_mode():
    outputs_path = Path(PROJECT_ROOT) / "outputs"
    if RUN_MODE == "archive_then_run":
        if outputs_path.exists() and any(outputs_path.iterdir()):
            backup_dir = Path(RUN_ARCHIVE_DIR) / "pre_run_outputs_backup"
            print(f"[RUN_MODE: archive_then_run] Backing up current outputs to {backup_dir}...")
            shutil.copytree(outputs_path, backup_dir, dirs_exist_ok=True)
            print("[SUCCESS] Backup complete.")
    elif RUN_MODE == "clean_run":
        print(f"[RUN_MODE: clean_run] Cleaning existing stage outputs under {outputs_path}...")
        for stage_dir in outputs_path.glob("stage-*"):
            if stage_dir.is_dir():
                shutil.rmtree(stage_dir)
        print("[SUCCESS] Stage outputs cleaned.")
    elif RUN_MODE == "preserve":
        print(f"[RUN_MODE: preserve] Preserving all existing experimental outputs.")

apply_run_mode()

def stage_artifacts_exist(stage_key: str) -> bool:
    if stage_key not in STAGE_METADATA:
        return False
    _, _, out_dir_name, key_files = STAGE_METADATA[stage_key]
    out_dir = Path(PROJECT_ROOT) / "outputs" / out_dir_name
    if not out_dir.exists():
        return False
    for kf in key_files:
        if not (out_dir / kf).exists():
            return False
    return True

def run_stage(stage_key: str) -> bool:
    if stage_key not in STAGE_METADATA:
        raise ValueError(f"Unknown stage key: '{stage_key}'. Valid stages: {STAGES}")
        
    module_name, stage_desc, out_folder, key_files = STAGE_METADATA[stage_key]
    stage_out_path = Path(PROJECT_ROOT) / "outputs" / out_folder
    
    # Check if stage skipping is requested
    if SKIP_COMPLETED_STAGES and stage_artifacts_exist(stage_key):
        print(f"\n[SKIP] Stage {stage_key} ({stage_desc}) output artifacts already exist. Skipping as SKIP_COMPLETED_STAGES=True.")
        return True
        
    print(f"\n" + "=" * 76)
    print(f" STARTING STAGE {stage_key}: {stage_desc.upper()}")
    print(f" Module: scripts/{module_name}.py | Output: outputs/{out_folder}/")
    print(f"=" * 76)
    
    # Pre-stage GPU memory inspection & cleanup for GPU-intensive stages
    if stage_key in GPU_INTENSIVE_STAGES:
        print(f"[STAGE {stage_key} GPU NOTICE] Computationally intensive stage. Preparing GPU resources...")
        clear_gpu_memory()
        print_gpu_stats()
        
    stage_log_file = Path(RUN_LOGS_DIR) / f"stage_{stage_key}_{RUN_ID}.log"
    
    t_start = time.time()
    current_status = load_run_status()
    current_status["stages"][stage_key] = {
        "status": "running",
        "start_time": datetime.now().isoformat(),
        "module": module_name
    }
    save_run_status(current_status)
    
    # Snapshot files prior to running to detect newly created artifacts
    pre_files = set()
    if stage_out_path.exists():
        pre_files = {p.name for p in stage_out_path.glob("**/*") if p.is_file()}
        
    try:
        # Import or reload module dynamically
        if module_name in sys.modules:
            mod = importlib.reload(sys.modules[module_name])
        else:
            mod = importlib.import_module(module_name)
            
        if not hasattr(mod, "main"):
            raise AttributeError(f"Module '{module_name}' has no main() function.")
            
        # Execute stage main function
        mod.main()
        
        elapsed = time.time() - t_start
        
        # Verify output directory and discover newly generated files
        post_files = set()
        if stage_out_path.exists():
            post_files = {p.name for p in stage_out_path.glob("**/*") if p.is_file()}
        new_files = post_files - pre_files
        all_stage_files = post_files if not pre_files else (new_files if new_files else post_files)
        
        print(f"\n[SUCCESS] Stage {stage_key} completed successfully in {elapsed:.2f} seconds.")
        print(f"Output Directory: outputs/{out_folder}/")
        print(f"Generated Files:  {sorted(list(all_stage_files))[:10]}")
        
        # Update persistent run status
        current_status["stages"][stage_key] = {
            "status": "success",
            "elapsed_seconds": round(elapsed, 2),
            "completed_at": datetime.now().isoformat(),
            "output_directory": f"outputs/{out_folder}",
            "generated_files": sorted(list(all_stage_files))
        }
        save_run_status(current_status)
        return True
        
    except Exception as e:
        elapsed = time.time() - t_start
        err_msg = traceback.format_exc()
        print(f"\n" + "!" * 76)
        print(f"[ERROR] Stage {stage_key} ({module_name}) FAILED after {elapsed:.2f} seconds:")
        print(err_msg)
        print("!" * 76)
        
        current_status["stages"][stage_key] = {
            "status": "failed",
            "elapsed_seconds": round(elapsed, 2),
            "failed_at": datetime.now().isoformat(),
            "error": str(e)
        }
        save_run_status(current_status)
        raise e

def run_pipeline_range(start_stage: str = None, end_stage: str = None):
    if start_stage is None:
        start_stage = RUN_FROM_STAGE or "01"
    if end_stage is None:
        end_stage = RUN_TO_STAGE or "18"
        
    if start_stage not in STAGES:
        raise ValueError(f"Invalid start_stage: '{start_stage}'. Valid: {STAGES}")
    if end_stage not in STAGES:
        raise ValueError(f"Invalid end_stage: '{end_stage}'. Valid: {STAGES}")
        
    start_idx = STAGES.index(start_stage)
    end_idx = STAGES.index(end_stage)
    
    if start_idx > end_idx:
        raise ValueError(f"start_stage '{start_stage}' must precede end_stage '{end_stage}'.")
        
    stage_sequence = STAGES[start_idx : end_idx + 1]
    print(f"\n======================================================================")
    print(f" EXECUTING PIPELINE RANGE: {start_stage} -> {end_stage}")
    print(f" Stages to execute ({len(stage_sequence)}): {stage_sequence}")
    print(f"======================================================================")
    
    t_range_start = time.time()
    for sk in stage_sequence:
        success = run_stage(sk)
        if not success:
            raise RuntimeError(f"Pipeline execution halted due to failure in Stage {sk}.")
            
    range_elapsed = time.time() - t_range_start
    print(f"\n======================================================================")
    print(f" [SUCCESS] COMPLETED STAGES {start_stage} TO {end_stage} IN {range_elapsed/60:.2f} MINUTES")
    print(f"======================================================================")

def show_stage_outputs(stage_key: str):
    if stage_key not in STAGE_METADATA:
        print(f"Unknown stage: {stage_key}")
        return
    _, desc, out_folder, _ = STAGE_METADATA[stage_key]
    out_dir = Path(PROJECT_ROOT) / "outputs" / out_folder
    print(f"\n--- Outputs for Stage {stage_key} ({desc}) ---")
    print(f"Directory: {out_dir}")
    if not out_dir.exists():
        print("  [Directory does not exist yet]")
        return
    files = list(out_dir.glob("**/*"))
    if not files:
        print("  [Directory is empty]")
        return
    for f in sorted(files):
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            print(f"  {f.name:<40} {size_kb:>10.2f} KB | {mtime}")

def inspect_stage(stage_key: str, preview_rows: int = 5):
    show_stage_outputs(stage_key)
    if stage_key not in STAGE_METADATA:
        return
    _, _, out_folder, key_files = STAGE_METADATA[stage_key]
    out_dir = Path(PROJECT_ROOT) / "outputs" / out_folder
    for kf in key_files:
        kp = out_dir / kf
        if kp.exists() and kp.is_file():
            print(f"\n[Preview of {kf}]:")
            try:
                if kf.endswith(".json"):
                    with open(kp, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if isinstance(data, dict):
                        keys_preview = list(data.keys())[:8]
                        print(f"  Keys ({len(data)} total): {keys_preview}")
                        for k in keys_preview[:3]:
                            print(f"  Sample [{k}]: {str(data[k])[:120]}")
                    elif isinstance(data, list):
                        print(f"  List containing {len(data)} items. First item preview:")
                        print(f"  {str(data[0])[:200]}")
                elif kf.endswith(".csv"):
                    df = pd.read_csv(kp, nrows=preview_rows)
                    print(df.to_string())
                elif kf.endswith(".jsonl"):
                    with open(kp, "r", encoding="utf-8") as f:
                        first_line = f.readline()
                        print(f"  First JSONL record:\n  {first_line[:200]}...")
                elif kf.endswith(".txt") or kf.endswith(".md"):
                    with open(kp, "r", encoding="utf-8") as f:
                        preview_text = f.read(300)
                    print(f"  {preview_text}...")
            except Exception as e:
                print(f"  Could not preview {kf}: {e}")

# Validate Resume Configuration if specified
if RESUME_FROM_STAGE is not None:
    if RESUME_FROM_STAGE not in STAGES:
        raise ValueError(f"Invalid RESUME_FROM_STAGE: '{RESUME_FROM_STAGE}'")
    resume_idx = STAGES.index(RESUME_FROM_STAGE)
    print(f"\n======================================================================")
    print(f" RESUME MODE VALIDATION: Starting from Stage {RESUME_FROM_STAGE}")
    print(f"======================================================================")
    if resume_idx > 0:
        prior_stage = STAGES[resume_idx - 1]
        prior_ok = stage_artifacts_exist(prior_stage)
        status_label = "PASS" if prior_ok else "FAIL"
        print(f"Resume validation:")
        print(f"Stage {prior_stage} outputs found: {status_label}")
        if not prior_ok:
            raise FileNotFoundError(
                f"Cannot resume from Stage {RESUME_FROM_STAGE}: Expected outputs from prior Stage {prior_stage} are missing!"
            )
    print(f"Starting from Stage {RESUME_FROM_STAGE}...")
    RUN_FROM_STAGE = RESUME_FROM_STAGE

## 8. Corpus Generation — Stages 01–10

This phase converts raw MARSIS relational tables into a clean, verified, multi-table maritime accident corpus:
* **Stage 01**: `01_parse_dictionary` — Parses MARSIS data dictionary, extracting field metadata & types.
* **Stage 02**: `02_profile_dataset` — Profiles column distributions, missingness, and cardinality.
* **Stage 03**: `03_discover_relationships` — Infers foreign keys and primary keys across relational tables.
* **Stage 04**: `04_select_semantic_columns` — Selects informative columns for downstream NLP generation.
* **Stage 05**: `05_merge_tables` — Performs schema reconciliation and multi-table joins.
* **Stage 05a**: `05a_validate_records` — Enforces domain constraints (speed, tonnage, crew bounds).
* **Stage 06**: `06_generate_documents` — Synthesizes narrative paragraphs using domain templates.
* **Stage 07**: `07_clean_documents` — Normalizes unicode, sanitizes artifacts, and deduplicates.
* **Stage 08**: `08_export_corpus` — Exports canonical `maritime_corpus.jsonl`, `maritime_corpus.txt`, and `manifest.json`.
* **Stage 09**: `09_statistics` — Generates corpus statistics and `corpus_quality_report.md`.
* **Stage 10**: `10_extract_vocabulary` — Extracts domain vocabulary and terminology frequencies.

In [ ]:
# Run Stages 01 through 10
# Respects configured RUN_FROM_STAGE and RUN_TO_STAGE
target_start = max(STAGES.index("01"), STAGES.index(RUN_FROM_STAGE)) if RUN_FROM_STAGE in STAGES else 0
target_end = min(STAGES.index("10"), STAGES.index(RUN_TO_STAGE)) if RUN_TO_STAGE in STAGES else STAGES.index("10")

if target_start <= target_end:
    start_key = STAGES[target_start]
    end_key = STAGES[target_end]
    print(f"Executing Corpus Generation Pipeline ({start_key} -> {end_key})...")
    run_pipeline_range(start_key, end_key)
else:
    print(f"Skipping Stages 01-10 because configured range is {RUN_FROM_STAGE} -> {RUN_TO_STAGE}.")

In [ ]:
# Inspect Key Artifacts from Corpus Generation (Stage 08 and Stage 09)
inspect_stage("08")
inspect_stage("09")

## 9. Knowledge / Representation Analysis — Stages 11–12

* **Stage 11**: `11_corpus_representations` — Generates multi-format representations (JSONL, TXT, CSV, Parquet, TSV) for downstream consumption.
* **Stage 12**: `12_semantic_importance` — Performs TF-IDF analysis, semantic domain informativeness scoring, K-Means clustering, and feature ablation.

In [ ]:
# Run Stages 11 through 12
target_start = max(STAGES.index("11"), STAGES.index(RUN_FROM_STAGE)) if RUN_FROM_STAGE in STAGES else STAGES.index("11")
target_end = min(STAGES.index("12"), STAGES.index(RUN_TO_STAGE)) if RUN_TO_STAGE in STAGES else STAGES.index("12")

if target_start <= target_end:
    start_key = STAGES[target_start]
    end_key = STAGES[target_end]
    print(f"Executing Knowledge & Representation Pipeline ({start_key} -> {end_key})...")
    run_pipeline_range(start_key, end_key)
else:
    print(f"Skipping Stages 11-12 because configured range is {RUN_FROM_STAGE} -> {RUN_TO_STAGE}.")

In [ ]:
# Inspect Knowledge Classification & Semantic Importance Outputs
inspect_stage("11")
inspect_stage("12")

## 10. Benchmarking — Stages 13–15

> **High Computational Demand Warning**:
> Stages 13–15 download Hugging Face models and execute GPU-intensive evaluation:
> - **Stage 13**: Tokenizer fertility, out-of-vocabulary (OOV) rate, and domain vocabulary coverage.
> - **Stage 14**: Multi-Model Masked Language Model (MLM) accuracy, cross-entropy loss, pseudo-log-likelihood (PLL), and domain-aware masking.
> - **Stage 15**: Cross-model throughput profiling, VRAM footprint, Pareto optimality analysis, and leaderboard generation.
> Ensure the Colab runtime is set to GPU (`Runtime` -> `Change runtime type` -> `GPU`).

In [ ]:
# GPU Status Check & Memory Cleanup before Benchmarking
print_gpu_stats()
clear_gpu_memory()

In [ ]:
# Run Stages 13 through 15
target_start = max(STAGES.index("13"), STAGES.index(RUN_FROM_STAGE)) if RUN_FROM_STAGE in STAGES else STAGES.index("13")
target_end = min(STAGES.index("15"), STAGES.index(RUN_TO_STAGE)) if RUN_TO_STAGE in STAGES else STAGES.index("15")

if target_start <= target_end:
    start_key = STAGES[target_start]
    end_key = STAGES[target_end]
    print(f"Executing Cross-Model Benchmarking Pipeline ({start_key} -> {end_key})...")
    run_pipeline_range(start_key, end_key)
else:
    print(f"Skipping Stages 13-15 because configured range is {RUN_FROM_STAGE} -> {RUN_TO_STAGE}.")

In [ ]:
# Inspect Benchmarking Artifacts (Stage 13, 14, 15)
inspect_stage("13")
inspect_stage("14")
inspect_stage("15")

## 11. Statistical Validation — Stages 16–17

* **Stage 16**: `16_statistical_analysis` — Executes pairwise Wilcoxon signed-rank tests, paired t-tests, bootstrap confidence intervals, Cohen's d effect sizes, and scoring feature ablation across model evaluations.
* **Stage 17**: `17_decision_engine` — Multi-Criteria Decision Analysis (MCDA), objective threshold scoring, model recommendation, and automated markdown research report generation.

In [ ]:
# Run Stages 16 through 17
target_start = max(STAGES.index("16"), STAGES.index(RUN_FROM_STAGE)) if RUN_FROM_STAGE in STAGES else STAGES.index("16")
target_end = min(STAGES.index("17"), STAGES.index(RUN_TO_STAGE)) if RUN_TO_STAGE in STAGES else STAGES.index("17")

if target_start <= target_end:
    start_key = STAGES[target_start]
    end_key = STAGES[target_end]
    print(f"Executing Statistical Validation & Decision Engine ({start_key} -> {end_key})...")
    run_pipeline_range(start_key, end_key)
else:
    print(f"Skipping Stages 16-17 because configured range is {RUN_FROM_STAGE} -> {RUN_TO_STAGE}.")

In [ ]:
# Inspect Statistical Significance & Final Decision Outputs
inspect_stage("16")
inspect_stage("17")

## 12. Corpus Linting — Stage 18

* **Stage 18**: `18_lint_corpus` — Automated corpus quality assurance. Verifies manifest integrity, checks for invalid nulls/empty texts, detects duplicate entries, audits unicode characters, and validates document structure.

In [ ]:
# Run Stage 18 (Corpus Linting)
if (RUN_FROM_STAGE is None or STAGES.index(RUN_FROM_STAGE) <= STAGES.index("18")) and \
   (RUN_TO_STAGE is None or STAGES.index(RUN_TO_STAGE) >= STAGES.index("18")):
    run_stage("18")
    inspect_stage("18")
else:
    print("Stage 18 excluded by configured execution range.")

## 13. Final Results Inspection

Comprehensive Experimental Results Dashboard.
Reads actual generated artifacts across Stages 13 through 18 and displays verified scientific metrics directly from disk (zero simulated or fabricated values):
* **Stage 13**: Tokenizer analysis outputs (`tokenizer_analysis.json`, `tokenizer_stage12_analysis.json`) and selected models (`selected_models.json`).
* **Stage 14**: Masked Language Model evaluation results (`bert_mlm_evaluation.json`), PLL results (`pll_results.json`), and focused domain-aware results (`focused_domain_aware_results.json`).
* **Stage 15**: Model Leaderboard (`leaderboard.csv`), Pareto frontier (`stage15_pareto.csv`), model comparison (`comparison.csv`), model profiles (`stage15_model_profiles.csv`), rankings (`stage15_rankings.csv`), and research report summary (`stage15_report.md`).
* **Stage 16**: Statistical significance outputs (`pairwise_tests.csv`, `stage16_pairwise.csv`), bootstrap results (`stage16_bootstrap.csv`), effect sizes, and ablation results (`ablation_study.json`, `stage16_ablation.csv`).
* **Stage 17**: Model selection outputs (`stage17_model_selection.csv`), decision summary (`decision_summary.json`), selection rationale, and research benchmark report (`benchmark_report.md`, `stage17_decision_report.md`).
* **Stage 18**: Corpus lint QA report (`corpus_lint_report.json`).

In [ ]:
import os
import glob
import json
import pandas as pd
from pathlib import Path

outputs_base = Path(PROJECT_ROOT) / "outputs"

print("=" * 80)
print(f"{'TSBC MARITIME PIPELINE — SCIENTIFIC BENCHMARK RESULTS DASHBOARD':^80}")
print("=" * 80)

# Check existence of stages 13 through 18
print("\n--- Output Directory Verification (Stages 13–18) ---")
for s_num in ["13", "14", "15", "16", "17", "18"]:
    s_dir = outputs_base / f"stage-{s_num}"
    exists = s_dir.exists()
    status = "EXISTS" if exists else "NOT FOUND"
    count = len(list(s_dir.glob("*"))) if exists else 0
    print(f"  Stage {s_num:<4} ({s_dir.name:<10}): [{status}] ({count} items)")

# --- STAGE 13: TOKENIZER ANALYSIS & SELECTED MODELS ---
print("\n" + "=" * 80)
print(" [STAGE 13] TOKENIZER BENCHMARK & SELECTED MODELS")
print("=" * 80)
s13_dir = outputs_base / "stage-13"
selected_models_file = s13_dir / "selected_models.json"
if selected_models_file.exists():
    try:
        with open(selected_models_file, "r", encoding="utf-8") as f:
            sm = json.load(f)
        print(f"Selected Models for Downstream Benchmarking ({len(sm)} total):")
        for m in sm:
            if isinstance(m, dict):
                print(f"  - {m.get('model_name', m.get('name', str(m)))}")
            else:
                print(f"  - {m}")
    except Exception as e:
        print(f"Error reading selected_models.json: {e}")

tok_analysis_file = s13_dir / "tokenizer_analysis.json"
if not tok_analysis_file.exists():
    tok_analysis_file = s13_dir / "tokenizer_stage12_analysis.json"
if tok_analysis_file.exists():
    try:
        with open(tok_analysis_file, "r", encoding="utf-8") as f:
            tdata = json.load(f)
        print(f"\nTokenizer Analysis Metrics Summary:")
        if isinstance(tdata, dict):
            for model_k, metrics in list(tdata.items())[:5]:
                if isinstance(metrics, dict):
                    fert = metrics.get("fertility", metrics.get("mean_fertility", "N/A"))
                    oov = metrics.get("oov_rate", metrics.get("unknown_rate", "N/A"))
                    cov = metrics.get("domain_coverage", metrics.get("vocab_coverage", "N/A"))
                    print(f"  {model_k:<32} | Fertility: {fert} | OOV Rate: {oov} | Domain Coverage: {cov}")
                else:
                    print(f"  {model_k}: {str(metrics)[:100]}")
    except Exception as e:
        print(f"Error reading tokenizer analysis: {e}")

# --- STAGE 14: MLM & PLL BENCHMARK MATRIX ---
print("\n" + "=" * 80)
print(" [STAGE 14] MASKED LANGUAGE MODEL (MLM) & PSEUDO-LOG-LIKELIHOOD (PLL)")
print("=" * 80)
s14_dir = outputs_base / "stage-14"
pll_file = s14_dir / "pll_results.json"
if pll_file.exists():
    try:
        with open(pll_file, "r", encoding="utf-8") as f:
            pll_data = json.load(f)
        print("PLL Benchmark Results:")
        if isinstance(pll_data, dict):
            for model_k, v in list(pll_data.items())[:6]:
                print(f"  {model_k:<32} | PLL Score / Metrics: {v}")
    except Exception as e:
        print(f"Error reading pll_results.json: {e}")

mlm_file = s14_dir / "bert_mlm_evaluation.json"
if mlm_file.exists():
    try:
        with open(mlm_file, "r", encoding="utf-8") as f:
            mlm_data = json.load(f)
        print("\nMLM Cross-Entropy & Perplexity Results:")
        if isinstance(mlm_data, dict):
            for model_k, v in list(mlm_data.items())[:6]:
                if isinstance(v, dict):
                    loss = v.get("loss", v.get("eval_loss", "N/A"))
                    ppl = v.get("perplexity", "N/A")
                    acc = v.get("accuracy", v.get("mlm_accuracy", "N/A"))
                    print(f"  {model_k:<32} | Loss: {loss} | Perplexity: {ppl} | Accuracy: {acc}")
                else:
                    print(f"  {model_k}: {str(v)[:100]}")
    except Exception as e:
        print(f"Error reading bert_mlm_evaluation.json: {e}")

focus_file = s14_dir / "focused_domain_aware_results.json"
if focus_file.exists():
    try:
        with open(focus_file, "r", encoding="utf-8") as f:
            fdata = json.load(f)
        print(f"\nFocused Domain-Aware Results ({len(fdata)} records found):")
        if isinstance(fdata, dict):
            for k, v in list(fdata.items())[:4]:
                print(f"  {k}: {str(v)[:120]}")
    except Exception as e:
        print(f"Error reading focused_domain_aware_results.json: {e}")

# --- STAGE 15: CROSS-MODEL BENCHMARKING & COMPUTATIONAL PROFILING ---
print("\n" + "=" * 80)
print(" [STAGE 15] CROSS-MODEL BENCHMARKING & LEADERBOARD")
print("=" * 80)
s15_dir = outputs_base / "stage-15"

# Leaderboard
lboard_file = s15_dir / "leaderboard.csv"
if lboard_file.exists():
    try:
        ldf = pd.read_csv(lboard_file)
        print("Model Leaderboard (leaderboard.csv):")
        print(ldf.to_string(index=False))
    except Exception as e:
        print(f"Error reading leaderboard.csv: {e}")

# Pareto Frontier
pareto_file = s15_dir / "stage15_pareto.csv"
if pareto_file.exists():
    try:
        pdf = pd.read_csv(pareto_file)
        print("\nPareto Frontier Models (stage15_pareto.csv):")
        print(pdf.to_string(index=False))
    except Exception as e:
        print(f"Error reading stage15_pareto.csv: {e}")

# Model Profiles
profiles_file = s15_dir / "stage15_model_profiles.csv"
if profiles_file.exists():
    try:
        pro_df = pd.read_csv(profiles_file)
        print(f"\nModel Profiles Summary ({len(pro_df)} models profiled):")
        print(pro_df.head(6).to_string(index=False))
    except Exception as e:
        print(f"Error reading stage15_model_profiles.csv: {e}")

# Comparison Table
comp_file = s15_dir / "comparison.csv"
if comp_file.exists():
    try:
        cdf = pd.read_csv(comp_file)
        print(f"\nModel Comparison Matrix ({len(cdf)} entries):")
        print(cdf.head(5).to_string(index=False))
    except Exception as e:
        print(f"Error reading comparison.csv: {e}")

# Stage 15 Markdown Report excerpt
rep15_file = s15_dir / "stage15_report.md"
if rep15_file.exists():
    try:
        with open(rep15_file, "r", encoding="utf-8") as f:
            content = f.read()
        print("\nStage 15 Executive Summary Excerpt:")
        print(content[:400] + "...\n")
    except Exception as e:
        print(f"Error reading stage15_report.md: {e}")

# --- STAGE 16: STATISTICAL SIGNIFICANCE & ABLATION ---
print("\n" + "=" * 80)
print(" [STAGE 16] STATISTICAL SIGNIFICANCE TESTING & FEATURE ABLATION")
print("=" * 80)
s16_dir = outputs_base / "stage-16"

boot_file = s16_dir / "stage16_bootstrap.csv"
if boot_file.exists():
    try:
        bdf = pd.read_csv(boot_file)
        print("Bootstrap Confidence Intervals (stage16_bootstrap.csv):")
        print(bdf.head(8).to_string(index=False))
    except Exception as e:
        print(f"Error reading stage16_bootstrap.csv: {e}")

pairwise_candidates = [s16_dir / "pairwise_tests.csv", s16_dir / "stage16_pairwise.csv"]
for pw_f in pairwise_candidates:
    if pw_f.exists():
        try:
            pw_df = pd.read_csv(pw_f)
            print(f"\nPairwise Significance Tests ({pw_f.name}):")
            print(pw_df.head(8).to_string(index=False))
            break
        except Exception as e:
            print(f"Error reading {pw_f.name}: {e}")

ablation_file = s16_dir / "ablation_study.json"
if not ablation_file.exists():
    ablation_file = s16_dir / "stage16_ablation.csv"
if ablation_file.exists():
    try:
        print(f"\nScoring Feature Ablation Results ({ablation_file.name}):")
        if ablation_file.suffix == ".json":
            with open(ablation_file, "r", encoding="utf-8") as f:
                abdata = json.load(f)
            for k, v in list(abdata.items())[:5]:
                print(f"  {k}: {v}")
        else:
            ab_df = pd.read_csv(ablation_file)
            print(ab_df.head(6).to_string(index=False))
    except Exception as e:
        print(f"Error reading ablation file: {e}")

# --- STAGE 17: OBJECTIVE THRESHOLD DECISION ENGINE ---
print("\n" + "=" * 80)
print(" [STAGE 17] DECISION ENGINE & RESEARCH REPORT")
print("=" * 80)
s17_dir = outputs_base / "stage-17"

dec_file = s17_dir / "decision_summary.json"
if dec_file.exists():
    try:
        with open(dec_file, "r", encoding="utf-8") as f:
            dsummary = json.load(f)
        print("Decision Summary (decision_summary.json):")
        rec_model = dsummary.get("selected_model", dsummary.get("recommended_model", "N/A"))
        print(f"  >> SELECTED / RECOMMENDED MODEL: {rec_model}")
        if "decision_rationale" in dsummary:
            print(f"  >> RATIONALE: {dsummary['decision_rationale']}")
        if "threshold_scores" in dsummary:
            print(f"  >> THRESHOLD SCORES: {dsummary['threshold_scores']}")
        if "criteria_weights" in dsummary:
            print(f"  >> CRITERIA WEIGHTS: {dsummary['criteria_weights']}")
    except Exception as e:
        print(f"Error reading decision_summary.json: {e}")

ms_file = s17_dir / "stage17_model_selection.csv"
if ms_file.exists():
    try:
        ms_df = pd.read_csv(ms_file)
        print("\nModel Selection Matrix (stage17_model_selection.csv):")
        print(ms_df.to_string(index=False))
    except Exception as e:
        print(f"Error reading stage17_model_selection.csv: {e}")

# Check for benchmark reports
report_candidates = [s17_dir / "stage17_decision_report.md", s17_dir / "benchmark_report.md"]
for r_file in report_candidates:
    if r_file.exists():
        try:
            with open(r_file, "r", encoding="utf-8") as f:
                rep_text = f.read()
            print(f"\n{r_file.name} Excerpt:")
            print(rep_text[:400] + "...\n")
            break
        except Exception as e:
            print(f"Error reading {r_file.name}: {e}")

# --- STAGE 18: CORPUS QUALITY LINTING ---
print("\n" + "=" * 80)
print(" [STAGE 18] AUTOMATED CORPUS QUALITY LINTING")
print("=" * 80)
s18_dir = outputs_base / "stage-18"
lint_file = s18_dir / "corpus_lint_report.json"
if lint_file.exists():
    try:
        with open(lint_file, "r", encoding="utf-8") as f:
            ldata = json.load(f)
        status_val = ldata.get("overall_status", ldata.get("status", "COMPLETED"))
        print(f"Corpus Lint Status: {status_val}")
        print(f"Total Documents Verified: {ldata.get('total_documents', 'N/A')}")
        if "checks" in ldata:
            for chk, res in ldata["checks"].items():
                print(f"  - {chk:<35}: {res}")
        elif "rule_evaluations" in ldata:
            for rk, rv in ldata["rule_evaluations"].items():
                print(f"  - {rk:<35}: {rv}")
    except Exception as e:
        print(f"Error reading corpus_lint_report.json: {e}")

print("\n" + "=" * 80)
print(f"{'END OF BENCHMARK RESULTS DASHBOARD':^80}")
print("=" * 80)

## 14. Archive and Reproducibility Package

Creates a publication-grade archival package:
1. Records complete hardware and software environment metadata into `experiment_metadata.json`.
2. Packages `outputs/`, `config/`, execution logs, and run metadata into a zip archive under `DRIVE_ROOT/archived_runs/`.
3. Excludes raw input datasets from the archive to keep size manageable.
4. Triggers optional direct download via Google Colab.

In [ ]:
import os
import sys
import json
import zipfile
import shutil
import platform
import subprocess
from datetime import datetime
from pathlib import Path

# --- 1. Gather Experiment Metadata ---
config_data = {}
cfg_path = Path(PROJECT_ROOT) / "config" / "config.json"
if cfg_path.exists():
    try:
        with open(cfg_path, "r", encoding="utf-8") as f:
            config_data = json.load(f)
    except Exception:
        pass

git_commit = "N/A"
try:
    res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True)
    if res.returncode == 0:
        git_commit = res.stdout.strip()
except Exception:
    pass

import torch
import transformers
import numpy as np
import pandas as pd

cuda_ver = torch.version.cuda if torch.cuda.is_available() else "N/A"
gpu_device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

status_record = load_run_status()

experiment_metadata = {
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "pytorch_version": torch.__version__,
    "cuda_version": cuda_ver,
    "gpu_device": gpu_device_name,
    "transformers_version": transformers.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "git_commit": git_commit,
    "project_root": PROJECT_ROOT,
    "run_mode": RUN_MODE,
    "stage_range_executed": f"{RUN_FROM_STAGE} -> {RUN_TO_STAGE}",
    "pipeline_config": config_data,
    "stage_execution_summary": status_record.get("stages", {})
}

meta_archive_path = Path(RUN_ARCHIVE_DIR) / "experiment_metadata.json"
with open(meta_archive_path, "w", encoding="utf-8") as f:
    json.dump(experiment_metadata, f, indent=2)
print(f"[SUCCESS] Experiment metadata recorded: {meta_archive_path}")

# --- 2. Build Archival ZIP Package ---
archive_zip_name = f"TSBC_MaritimePipeline_results_{RUN_ID}.zip"
archive_zip_path = Path(DRIVE_ROOT) / "archived_runs" / archive_zip_name

print(f"\nBuilding archival package: {archive_zip_path}...")
outputs_path = Path(PROJECT_ROOT) / "outputs"
config_path = Path(PROJECT_ROOT) / "config"
logs_path = Path(PROJECT_ROOT) / "outputs" / "logs"

with zipfile.ZipFile(archive_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    # 1. Add experiment metadata and status
    if meta_archive_path.exists():
        zipf.write(meta_archive_path, arcname="run_metadata/experiment_metadata.json")
    if status_file_archive.exists():
        zipf.write(status_file_archive, arcname="run_metadata/run_status.json")
        
    # 2. Add config directory
    if config_path.exists():
        for cp in config_path.glob("**/*"):
            if cp.is_file():
                arcname = str(Path("config") / cp.relative_to(config_path))
                zipf.write(cp, arcname=arcname)
                
    # 3. Add outputs directory (excluding any raw data)
    if outputs_path.exists():
        for op in outputs_path.glob("**/*"):
            if op.is_file():
                arcname = str(Path("outputs") / op.relative_to(outputs_path))
                zipf.write(op, arcname=arcname)
                
    # 4. Add execution run logs
    if os.path.exists(RUN_LOGS_DIR):
        for lp in Path(RUN_LOGS_DIR).glob(f"*{RUN_ID}*"):
            if lp.is_file():
                zipf.write(lp, arcname=f"run_logs/{lp.name}")

archive_size_mb = archive_zip_path.stat().st_size / (1024 * 1024)
print(f"[SUCCESS] Archival package created successfully!")
print(f"File: {archive_zip_path}")
print(f"Size: {archive_size_mb:.2f} MB")

# --- 3. Optional Direct Colab Download ---
print("\nTo download the results archive to your local computer:")
try:
    from google.colab import files
    print("Initiating Colab browser download...")
    files.download(str(archive_zip_path))
except ImportError:
    print(f"Direct download is only available in Google Colab.")
    print(f"The archive is safely persisted on Google Drive at: {archive_zip_path}")